<a href="https://www.kaggle.com/code/ahmad03038/cristiano-ronaldo-youtube-stats-daily-pull?scriptVersionId=354265499" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Import necessary libraries

In [1]:
from googleapiclient.discovery import build
import pandas as pd
import os
import shutil
from datetime import datetime, timezone
import re
from kaggle_secrets import UserSecretsClient  # Securely access API key

<h2 style="color: #8B0000;">  Find The Youtube Channel ID of Cristiano Ronaldo Using Youtube API: </h2>


In [2]:
from googleapiclient.discovery import build
from kaggle_secrets import UserSecretsClient

def get_channel_id(api_key, username):
    youtube = build("youtube", "v3", developerKey=api_key)
    request = youtube.channels().list(
        part="id",
        forHandle=username  #  "@cristiano"
    )
    response = request.execute()
    
    if "items" in response and len(response["items"]) > 0:
        channel_id = response["items"][0]["id"]
        print(f"✅ Channel ID for {username}: {channel_id}")
        return channel_id
    else:
        print("❌ Channel not found!")
        return None

#  Replace with your YouTube API key

user_secrets = UserSecretsClient()
API_KEY_AHMED = user_secrets.get_secret("YOUTUBE_API_KEY_AHMED")
API_KEY = API_KEY_AHMED

#  Find Cristiano Ronaldo's Channel ID
channel_id = get_channel_id(API_KEY, "@cristiano")


✅ Channel ID for @cristiano: UCtxD0x6AuNNqdXO9Wp5GHew


# Daily Updates (Pull of Data)

<h2 style="color: #8B0000;"> Step 1: Authenticate & Connect to YouTube API</h2>


In [3]:
def get_youtube_api():
    """
    Retrieves the YouTube API key securely from Kaggle Secrets and initializes the API client.
    """
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("YOUTUBE_API_KEY_AHMED")  # Secure API Key
    youtube = build("youtube", "v3", developerKey=API_KEY)
    return youtube

<h2 style="color: #8B0000;">   Step 2: Fetch Video IDs from Cristiano Ronaldo's YouTube Channel</h2>


In [4]:
def get_channel_videos(youtube, channel_id, max_results=1000000):
    """
    Fetches the latest video IDs from a given YouTube channel.
    
    Parameters:
        youtube (object): YouTube API client
        channel_id (str): ID of the YouTube channel
        max_results (int): Maximum number of videos to fetch
    
    Returns:
        list: Video IDs from the channel
    """
    video_ids = []
    next_page_token = None
    
    while True:
        request = youtube.search().list(
            part="id",
            channelId=channel_id,
            maxResults=min(max_results - len(video_ids), 50), 
            order="date",  # Get the latest videos
            type="video",
            pageToken=next_page_token
        )
        response = request.execute()
        
        video_ids.extend([item["id"]["videoId"] for item in response.get("items", [])])
        
        next_page_token = response.get("nextPageToken")
        if not next_page_token or len(video_ids) >= max_results:
            break
    
    return video_ids


<h2 style="color: #8B0000;">  Step 3: Fetch Video Details (Views, Likes, Comments, etc.)</h2>


In [5]:

def get_video_details(youtube, video_ids):
    """
    Retrieves detailed statistics and metadata for a list of video IDs.
    
    Parameters:
        youtube (object): YouTube API client
        video_ids (list): List of video IDs
    
    Returns:
        list: Video statistics and metadata
    """
    video_stats = []
    for i in range(0, len(video_ids), 50):  # Fetch stats in batches of 50
        request = youtube.videos().list(
            part="snippet,statistics,contentDetails,topicDetails,localizations",
            id=",".join(video_ids[i:i+50])
        )
        response = request.execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content_details = item.get("contentDetails", {})
            topic_details = item.get("topicDetails", {})
            localizations = item.get("localizations", {}).get("en", {})

            duration_seconds = convert_duration(content_details.get("duration", ""))
            video_type = "long" if duration_seconds >= 60 else "short"

            video_stats.append({
                "id": item["id"],
                "title": snippet.get("title", "N/A"),
                "description": snippet.get("description", "N/A"),
                "publishTime": snippet.get("publishedAt", "N/A"),
                "kind_stats": item.get("kind", "N/A"),
                "duration_seconds": duration_seconds,
                "video_type": video_type,
                "viewCount": statistics.get("viewCount", "N/A"),
                "likeCount": statistics.get("likeCount", "N/A"),
                "commentCount": statistics.get("commentCount", "N/A"),
                "thumbnails.default.url": snippet.get("thumbnails", {}).get("default", {}).get("url", "N/A"),
                "thumbnails.medium.url": snippet.get("thumbnails", {}).get("medium", {}).get("url", "N/A"),
                "thumbnails.high.url": snippet.get("thumbnails", {}).get("high", {}).get("url", "N/A"),
                "contentDetails.duration": content_details.get("duration", "N/A"),
                "contentDetails.dimension": content_details.get("dimension", "N/A"),
                "topicDetails.topicCategories": topic_details.get("topicCategories", "N/A"),
                "snippet.defaultLanguage": snippet.get("defaultLanguage", "N/A"),
                "localizations.en.title": localizations.get("title", "N/A"),
                "localizations.en.description": localizations.get("description", "N/A"),
                "snippet.tags": snippet.get("tags", "N/A"),
                "pull_date": datetime.now(timezone.utc).date().isoformat()  # Timestamp when data is pulled
            })
    return video_stats


<h2 style="color: #8B0000;"> Step 4: Convert YouTube Video Duration to Seconds</h2>


In [6]:
def convert_duration(duration):
    """
    Converts YouTube video duration format (ISO 8601) to total seconds.
    
    Parameters:
        duration (str): ISO 8601 duration string (e.g., PT5M20S)
    
    Returns:
        int: Total duration in seconds
    """
    pattern = re.compile(r'PT(\d+H)?(\d+M)?(\d+S)?')
    matches = pattern.match(duration)
    if not matches:
        return 0
    hours = int(matches.group(1)[:-1]) if matches.group(1) else 0
    minutes = int(matches.group(2)[:-1]) if matches.group(2) else 0
    seconds = int(matches.group(3)[:-1]) if matches.group(3) else 0
    return hours * 3600 + minutes * 60 + seconds

<h2 style="color: #8B0000;"> Step 5: Save Data (Appending Latest Stats to Existing Dataset)</h2>


In [7]:
def save_stats(stats, filename="cristiano_youtube_stats.csv"):
    """
    Saves or updates the YouTube dataset by appending new video statistics.
    
    Parameters:
        stats (list): Video statistics data
        filename (str): CSV filename to save the data
    
    Returns:
        None
    """
    existing_file = f"../input/cristiano-youtube-stats-daily/{filename}"
    save_path = os.path.join("/kaggle/working/", filename)
    
    # Step 5.1: Copy Existing Data (if available)
    if os.path.exists(existing_file):
        shutil.copy(existing_file, save_path)
        df_existing = pd.read_csv(save_path)
    else:
        df_existing = pd.DataFrame()
    
    # Step 5.2: Append Latest Stats and Remove Duplicates
    df_new = pd.DataFrame(stats)
    df_combined = pd.concat([df_existing, df_new], ignore_index=True).drop_duplicates(subset=["id", "pull_date"], keep="last")
    
    # Step 5.3: Save the Updated Dataset
    df_combined.to_csv(save_path, index=False)
    print(f"✅ Data updated successfully at: {save_path}")

<h2 style="color: #8B0000;"> Step 6: Main Execution</h2>


In [8]:
def main():
    """
    Main function to execute the script:
    - Connects to YouTube API
    - Fetches video IDs from Cristiano Ronaldo's channel
    - Retrieves video details
    - Saves the updated dataset
    """
    youtube = get_youtube_api()
    channel_id = "UCtxD0x6AuNNqdXO9Wp5GHew"  # Cristiano Ronaldo's YouTube Channel ID

    print("📡 Fetching all available videos...")
    video_ids = get_channel_videos(youtube, channel_id, max_results=100000)
    
    if not video_ids:
        print("⚠️ No videos found!")
        return

    print(f"📊 Fetching details for {len(video_ids)} videos...")
    stats = get_video_details(youtube, video_ids)

    if stats:
        save_stats(stats)
        print("🎯 YouTube stats fetched and saved successfully!")
    else:
        print("❌ Failed to fetch video stats.")



<h2 style="color: #8B0000;">Step 7: Run the  main function of the Script</h2>


In [9]:
if __name__ == "__main__":
    main()

📡 Fetching all available videos...
📊 Fetching details for 98 videos...
✅ Data updated successfully at: /kaggle/working/cristiano_youtube_stats.csv
🎯 YouTube stats fetched and saved successfully!
